# Notebook 11 — Overnight full JEPA workflow

Tagged definitions only. Six fresh paired seeds, three encoders, timing-only longer schedule, initial/350/final frozenEMAprobes. Run All in main launches real detached training plus independent deadline and hourly agent health supervisor. Outputs visualHTML/PNGs/reviewnotebook and fullresumable states. Original12hour cap includes allrecovery. Resume unchangedsameRUN_IDbefore originaldeadline; invalidstates fail. Official test remainsreserved. Checkpoints: checkpoints/<run_id>/.

In [ ]:
"""Notebook11 orchestration: one persisted 12-hour experimental clock."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
from datetime import datetime, timezone
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
MAIN_NOTEBOOK='notebooks/11_Overnight_Manifold_JEPA.ipynb'
TERMINAL={'COMPLETED','FAILED','INCOMPLETE'}

def _read(path):return json.loads(Path(path).read_text())
def _normalized(value):return json.loads(json.dumps(value,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(_normalized(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _export_hash(path):
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in _read(path)['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()
def _load(root,name):
    key='fmm11_'+name
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/11_manifold_long'/(name+'.ipynb')))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def source_hashes(root):
    return {str(p.relative_to(root)):_export_hash(p) for folder in ('fmm_jepa','experiments/04_followup','experiments/05_scaling','experiments/07_energy','experiments/08_replication','experiments/09_deep_energy','experiments/10_manifold_kernel','experiments/11_manifold_long') for p in sorted((Path(root)/folder).glob('*.ipynb'))}

def consumer_hash(root):return _hash([''.join(c['source']) for c in _read(Path(root)/MAIN_NOTEBOOK)['cells'] if c['cell_type']=='code'])

def study_configuration(run_id='manifold-long-001'):
    if not run_id or any(not(c.isalnum() or c in '-_') for c in run_id):raise ValueError('Safe nonempty run ID required')
    return {'schema':'fmm-jepa-notebook11-study/v1','run_id':run_id,'hard_limit_seconds':43200,
        'device':'cuda:0','dtype':'float32','image_size':128,'effective_batch':128,'microbatch':128,
        'seeds':list(range(20261401,20261407)),'variants':['vit','spatial_kernel','learned_kernel'],
        'update_candidates':[2400,1800,1200,1000],'intermediate_updates':350,
        'round_updates':50,'probe_epochs':20,'heldout_enabled':False,
        'input_pipeline':{'workers':6},'execution_optimization':True,
        'kernel_geometry':{'depth':4,'heads':3,'coordinate_dim':8,'bandwidths':[.15,.35,.75],'content_scale':.5,'checkpoint':False},
        'analysis':{'role':'prospectively_registered_primary_with_exploratory_secondary','n_pairs':6,'df':5,'confidence_level':.95,
            'contrasts':['learned_kernel_minus_vit','learned_kernel_minus_spatial_kernel','spatial_kernel_minus_vit'],
            'endpoint':'Fixed epoch20 frozen-EMA-probe development top1 at final common successful-update budget, percentage points',
            'interpretation':'Six fresh paired training seeds; primary learned-minus-ViT final difference. Paired t interval and exact sign test; no guarantee of significance, universal superiority or novelty. Conditional on reused development membership.'},
        'primary_analysis':{'role':'registered_primary','n_pairs':6,'df':5,'alpha':.05,
            'contrast':'learned_kernel_minus_vit_final','tests':['paired_t_two_sided','exact_two_sided_sign'],
            'endpoint':'Final epoch20 frozen-EMA-probe development top1; final update count selected from timing alone before quality outcomes.'},
        'secondary_analysis':'Intermediate350accuracy, initialization-adjusted gains, learned-minus-spatial, spatial-minus-ViT, geometry and compute are exploratory.',
        'budget_policy':'Fixed6freshpairs x3architectures. Pick highest candidate fitting original12h after discarded calibration with10percent timing margin,3600s evaluation/visualization and1800s preflight/recovery reserve. No quality-based change, cap extension or missing-arm substitution.',
        'selection':'Sixseeds/threearchitectures/epoch20probes at0,350,final fixed before quality; final updates selected by timing-only feasibility. No winner-based stopping.',
        'interpretation':'Full custom JEPA pipeline with longer learning and more paired seeds. Exact dense kernel sums; no FMM speedup or manifold-isomorphism claim. Official test reserved.',
        'heartbeat':{'interval_seconds':3600,'poll_seconds':30,'stall_seconds':1200,
            'automatic_resume_attempts':3,'agent_review_enabled':True,'agent_timeout_seconds':900},
        'visual_review':'notebooks/11_Overnight_Results_Review.ipynb'}



def _status(artifact,stage,status,**details):
    result={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',result)
    print(stage+': '+status,flush=True)
    return result

def _validation(root):
    root=Path(root);folder=root/'artifacts/notebook_11_validation'
    result={}
    for stem,files in [('operator',['encoder']),('input_pipeline',['input_pipeline']),('runner_evaluation',['runner','evaluation']),('execution_optimization',['engineering']),('heartbeat',['heartbeat']),('visual_report',['visualization'])]:
        p=folder/(stem+'_validation.json');report=_read(p)
        if report.get('status')!='PASS':raise ValueError('Notebook11 needs PASS '+stem+' validation')
        for name in files:
            actual=_export_hash(root/'experiments/11_manifold_long'/(name+'.ipynb'))
            if report.get(name+'_export_sha256')!=actual:raise ValueError('Validation source differs: '+name)
        result[stem]={'path':str(p.relative_to(root)),'sha256':digest_file(p)}
    agent=folder/'agent_probe/probe_result.json';report=_read(agent)
    if report.get('status')!='PASS':raise ValueError('Hourly agent must pass authentication/tool probe before launch')
    result['hourly_agent']={'path':str(agent.relative_to(root)),'sha256':digest_file(agent)}
    return result



def _check_binding(root,binding,config):
    reference='artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json'
    if binding.get('reference_pilot_plan')!={'path':reference,'sha256':digest_file(Path(root)/reference)}:
        raise ValueError('The pilot configuration source changed')
    if binding['configuration']!=_normalized(config):raise ValueError('Registered run configuration changed')
    if binding['source_hashes']!=source_hashes(root):raise ValueError('Registered source definitions changed; use a new run ID')
    if binding['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Registered main notebook code changed')
    if binding['operator_validation']!=_validation(root):raise ValueError('Bound validation evidence changed')

def _clock(artifact,binding):
    result=_read(Path(artifact)/'budget_clock.json')
    if result!=binding['budget_clock'] or result['max_seconds']!=43200:raise ValueError('Original 12-hour clock changed')
    return result

def _prepare(root,config):
    if _normalized(config)!=_normalized(study_configuration(config['run_id'])):
        raise ValueError('This pilot has a fixed registered configuration; no seed/update/architecture edits are permitted')
    root=Path(root).resolve();artifact=root/'artifacts/notebook_11'/config['run_id'];artifact.mkdir(parents=True,exist_ok=True)
    path=artifact/'study_binding.json'
    if path.exists():
        binding=_read(path);_check_binding(root,binding,config)
    else:
        if (artifact/'budget_clock.json').exists() or (artifact/'stages').exists():raise ValueError('Unbound existing experiment evidence; refusing restart')
        validation=_validation(root);now=time.time()
        clock={'started_unix':now,'deadline_unix':now+43200,'max_seconds':43200,'policy':'Original launch clock includes calibration, recovery verification, training and evaluation; never resets.'}
        binding={'schema':'notebook11-binding/v1','configuration':_normalized(config),'source_hashes':source_hashes(root),
                 'consumer_notebook_sha256':consumer_hash(root),'budget_clock':clock,'operator_validation':validation,
                 'reference_pilot_plan':{'path':'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json',
                    'sha256':digest_file(root/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')}}
        atomic_json(path,binding);atomic_json(artifact/'budget_clock.json',clock)
    return artifact,binding,_clock(artifact,binding)

def _choose_budget(config,calibration,deadline):
    if calibration['selected_microbatch']!=128 or calibration['selected_accumulation_steps']!=1:
        raise ValueError('Physical/effective batch128 is fixed')
    timings=calibration['selected_model_step_seconds']
    if set(timings)!=set(config['variants']):raise ValueError('All registered variants need calibration')
    if any(not isinstance(t,(int,float)) or not(0<t<float('inf')) for t in timings.values()):raise ValueError('Invalid timings')
    per_pair=sum(timings.values());remaining=deadline-time.time();count=len(config['seeds'])
    reserve=3600;preflight_recovery=1800;choices=[]
    for updates in config['update_candidates']:
        predicted=1.10*count*updates*per_pair
        choices.append({'updates':updates,'predicted_training_seconds':predicted,
            'predicted_total_remaining_seconds':predicted+reserve+preflight_recovery})
    selected=next((x for x in choices if x['predicted_total_remaining_seconds']<=remaining),None)
    if selected is None:raise TimeoutError('No complete six-seed/three-arm candidate fits the original12-hour cap; refusing a smaller study: '+str(choices))
    return {'status':'PASS',**selected,'seeds':count,'architectures':len(config['variants']),
        'evaluation_reserve_seconds':reserve,'preflight_recovery_reserve_seconds':preflight_recovery,
        'remaining_seconds':remaining,'calibration_round_seconds':per_pair,'timing_only_candidates':choices,
        'selection_rule':'Largest predeclared candidate fitting the original absolute clock; no quality selection.'}


def _plan(root,study,binding,bundle,base,calibration,budget):
    preparation=_load(root,'preparation');arms=[];seeds=study['seeds'];updates=budget['updates']
    for index,seed in enumerate(seeds):
        for variant in study['variants']:
            cfg=preparation.arm_config(base,variant)
            cfg['kernel11'].update(copy.deepcopy(study['kernel_geometry']));cfg['kernel11']['variant']=variant
            cfg['training'].update(updates=updates,microbatch=calibration['selected_microbatch'],accumulation_steps=calibration['selected_accumulation_steps'])
            cfg['seeds'].update(model_init=seed,data_order=941101+1000*index,augmentation=952202+1000*index,mask=963303+1000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            cfg['evaluation'].update(batch_size=min(64,calibration['selected_microbatch']),diagnostic_steps=[0,study['intermediate_updates'],updates])
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            cfg['input_pipeline']=copy.deepcopy(study['input_pipeline'])
            cfg['execution_optimization']=study['execution_optimization']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit' if variant=='vit' else 'semantic','variant':variant,'seed':seed,'condition':variant,'config':cfg})
    return _normalized({'schema':'notebook11-stage-plan/v1','run_id':study['run_id'],'stage_id':'learning','stage_role':'learning',
         'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':updates,'milestones':[study['intermediate_updates'],updates],
         'round_updates':study['round_updates'],'effective_batch':128,'seeds':seeds,'variants':study['variants'],'arms':arms,
         'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
         'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
         'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-overnight-manifold','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
         'study_configuration_hash':_hash(study),'numerical_backend':'dense-learned-kernel-v1','heldout_status':'NOT_RUN',
         'analysis':study['analysis'],'primary_analysis':study['primary_analysis'],'secondary_analysis':study['secondary_analysis'],
         'excluded_pilot_seeds':[20261011,20261012,20261201,20261202,20261301,20261302]+list(range(20261101,20261111)),
         'selection':study['selection'],'capacity_policy':'Shared stem/allfourMLPs/norms/predictor; kernel controls share value/projection initialization; active counts differ.',
         'failure_policy':'Stop first numerical failure; never skip samples.','budget_selection':budget})

def run_study(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_11'/run_id
    binding=_read(artifact/'study_binding.json');study=binding['configuration'];_check_binding(root,binding,study)
    clock=_clock(artifact,binding);deadline=clock['deadline_unix']
    if (artifact/'study_status.json').exists() and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
    if time.time()>=deadline:raise TimeoutError('Original 12-hour deadline expired')
    preparation,runner,evaluation=(_load(root,n) for n in ('preparation','runner','evaluation'))
    with run_writer(artifact):
        try:
            folder=artifact/'stages/learning';folder.mkdir(parents=True,exist_ok=True)
            (folder/'predictions').mkdir(exist_ok=True)
            base=preparation.base_config(root)
            base['kernel11'].update(copy.deepcopy(study['kernel_geometry']))
            base['input_pipeline']=copy.deepcopy(study['input_pipeline'])
            base['execution_optimization']=study['execution_optimization']
            _status(artifact,'verify_training_and_development_data','RUNNING')
            bundle=preparation.prepare_bundle(root,folder,base)
            _status(artifact,'actual_update_calibration','RUNNING')
            calibration=preparation.calibrate(root,folder,base,bundle,deadline)
            plan_path=folder/'plan.json'
            if plan_path.exists():
                plan=_read(plan_path)
                if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Saved plan sources changed')
                if plan['study_configuration_hash']!=_hash(study):raise ValueError('Saved plan study changed')
            else:
                budget=_choose_budget(study,calibration,deadline)
                plan=_plan(root,study,binding,bundle,base,calibration,budget)
                preparation.immutable(plan_path,plan)
            _status(artifact,'exact_resume_preflight','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            runner.prepare_stage(root,folder,plan,bundle)
            preflight=runner.verify_stage_resume(root,folder,plan,bundle,deadline)
            atomic_json(folder/'preflight_result.json',preflight)
            if preflight.get('status')!='PASS':raise RuntimeError('Exact-resume preflight did not pass')
            _status(artifact,'evaluation_integration_preflight','RUNNING')
            scoring=evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline)
            atomic_json(folder/'evaluation_preflight_result.json',scoring)
            if scoring.get('status')!='PASS':raise RuntimeError('Evaluation integration preflight failed')
            _check_binding(root,binding,study)
            _status(artifact,'paired_training','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            training_path=folder/'training_result.json'
            if training_path.exists() and _read(training_path).get('status')=='COMPLETE':
                # Preserve the authoritative training record and evaluation identity on evaluation-only recovery.
                training=_read(training_path)
                evaluation._stage_jobs(plan,training)
                from fmm_jepa.food101_training import validate_training_manifest
                for arm in plan['arms']:
                    directory=root/'checkpoints'/plan['run_id']/'stages/learning/arms'/arm['id']
                    checkpoint=validate_training_manifest(directory)
                    if checkpoint['successful_updates']!=plan['updates'] or checkpoint['attempted_updates']!=plan['updates']:
                        raise ValueError('Completed training record/checkpoint differs')
                    runner.validate_stage_milestones(directory,plan,required=plan['milestones'])
            else:
                training=runner.train_stage(root,folder,plan,bundle,deadline)
                atomic_json(training_path,training)
            _status(artifact,'frozen_feature_evaluation','RUNNING',updates=plan['updates'],seeds=plan['seeds'])
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            visual=_load(root,'visualization').build_visual_report(root,folder,plan,bundle,result,deadline_unix=deadline)
            result['visual_review']=visual
            atomic_json(artifact/'study_results.json',result)
            if result.get('status') not in ('PASS','COMPLETE','COMPLETED'):raise TimeoutError('Evaluation incomplete under original deadline')
            _check_binding(root,binding,study)
            atomic_json(folder/'stage_evaluation.json',result)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],heldout_status='NOT_RUN',results=str(folder/'stage_results.md'))
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),cause=repr(error.__cause__) if error.__cause__ else None,elapsed_seconds=time.time()-clock['started_unix'])
            raise

def _exists(session):return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_study(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_11'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        artifact,binding,clock=_prepare(root,config);session='fmm-manifold-long-'+config['run_id']
        if _exists(session):
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            _load(root,'heartbeat').launch_heartbeat(root,config['run_id'])
            return {'status':'EXISTING_SESSION','session':session,'deadline_unix':clock['deadline_unix'],'artifact':str(artifact)}
        prior=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
        if prior.get('status')=='COMPLETED':return prior
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Original deadline expired; Run All cannot reset it')
        setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-overnight-manifold' and 'ahr-export' in c.metadata.get('tags',[]))
        code=setup+'\nWORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')\n'
        worker=artifact/'worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook11 detached worker\nOutputs: this run folder; full-state checkpoints: checkpoints/'+config['run_id']+'/. Resume with unchanged Notebook11 Run All before the original 12-hour deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
        launch={'session':session,'artifact':str(artifact),'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':clock['deadline_unix'],'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'binding_hash':_hash(binding)}
        atomic_json(artifact/'launch.json',launch);_status(artifact,'launching','STARTING')
        try:
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            _load(root,'heartbeat').launch_heartbeat(root,config['run_id'])
            subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
        except BaseException as error:_status(artifact,'launch_failed','FAILED',error=repr(error));raise
        return {'status':'LAUNCHED',**launch}

def inspect_study(root,run_id):
    artifact=Path(root)/'artifacts/notebook_11'/run_id
    if not (artifact/'study_status.json').exists():return {'status':'NOT_LAUNCHED'}
    status=_read(artifact/'study_status.json');clock=_read(artifact/'budget_clock.json')
    result={**status,'session':'fmm-manifold-long-'+run_id,'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'remaining_minutes':max(0,(clock['deadline_unix']-time.time())/60)}
    rows=[]
    for p in sorted((Path(root)/'checkpoints'/run_id/'stages/learning').glob('arms/*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',m.get('latest_step'))})
    result['saved_training_progress']=rows
    return result


def readiness(root,config):
    """Read-only prelaunch check; creates no run folder, budget clock or tmux worker."""
    root=Path(root).resolve()
    if _normalized(config)!=_normalized(study_configuration(config['run_id'])):raise ValueError('Registered configuration changed')
    validation=_validation(root)
    encoder=_load(root,'encoder');preparation=_load(root,'preparation')
    base=preparation.base_config(root);rows=[]
    from fmm_jepa.food101_training import isolated_rng
    for variant in config['variants']:
        cfg=copy.deepcopy(base['model']);cfg['kernel11']={**config['kernel_geometry'],'variant':variant}
        with isolated_rng(base['seeds']['model_init']):model=encoder.build_model(cfg,kind='vit' if variant=='vit' else 'semantic')
        counts=encoder.parameter_counts(model)
        total=counts['encoder_total'];inactive=counts['encoder_inactive_scaffold']
        rows.append({'architecture':variant,'depth':len(model.context.blocks),'encoder_parameters_total':total,
            'encoder_parameters_active':total-inactive,'inactive_parameters':inactive,
            'trainable_parameters_total':sum(p.numel() for p in model.parameters() if p.requires_grad)})
        del model
    artifact=root/'artifacts/notebook_11'/config['run_id']
    started=(artifact/'budget_clock.json').exists()
    current=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
    scientific='COMPLETE' if current.get('status')=='COMPLETED' else ('NOT_RUN' if not started else current.get('status','IN_PROGRESS'))
    return {'status':'EXISTING_STUDY' if started else 'READY_TO_LAUNCH','production_run_started':started,'expected_runtime':'Up to12hours including preflight,training,54probes,visualreport and recovery; timing-selected complete schedule.',
        'architecture_summary':rows,'validation':validation,'run_id':config['run_id'],'planned_training_runs':18,'planned_probe_evaluations':54,
        'checkpoint_location':str(root/'checkpoints'/config['run_id']), 'scientific_results':scientific}
